In [ ]:
from openai import OpenAI
from huggingface_hub import login
from dotenv import load_dotenv
import os
import json
from datasets import load_dataset
from _2_2__rag_question_answering import rag_answer

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

openai = OpenAI()

In [ ]:
ds = load_dataset("ayanmukherjee/repliqa-4-company-policies-answerable")

In [ ]:
# Fine-tuned model name obtained from OpenAI API Platforms.
OPENAI_FINE_TUNED_MODEL = "ft:gpt-4.1-nano-2025-04-14:personal:company-policy-chatbot:DYDYKAKZ"

def closed_source_answer(question):
    messages = [{"role": "user", "content": question}]
    response = openai.chat.completions.create(model= OPENAI_FINE_TUNED_MODEL, messages = messages)
    return response.choices[0].message.content

In [ ]:
with open("responses/closed_source_responses.jsonl", "w") as f:
    for item in ds["test"]:
        question = item["question"]
        answer = item["long_answer"]
        model_response = closed_source_answer(question=question)
        data = {
            "question": question,
            "answer": answer,
            "model_response": model_response
        }
        f.write(json.dumps(data) + '\n')


In [ ]:
with open("responses/rag_responses.jsonl", "w") as f:
    for item in ds["test"]:
        question = item["question"]
        answer = item["long_answer"]
        model_response = rag_answer(question=question)
        data = {
            "question": question,
            "answer": answer,
            "model_response": model_response
        }
        f.write(json.dumps(data) + '\n')
